# Whole-Blob Hybrid vs Segmented Comparison

This notebook compares three validation strategies on the same files:

- `blob_parser`: whole ROI OCR once, then parse raw OCR text directly
- `blob_recovered`: whole ROI OCR once, then recover ordered lines from visual row count before parsing
- `segmented_pipeline`: the current segmented line pipeline used for comparison

The ROI detection path is the same as the walkthrough: `TopLeftBlueGrayBoxDetector`. The difference is only what happens **after** the ROI is cropped.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_repo(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "app" / "pipeline" / "echo_ocr_pipeline.py").is_file():
            return path
    raise RuntimeError("Run this notebook from the Master repo.")


REPO = find_repo(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("REPO:", REPO)

REPO: /home/warre/Documents/howest/Semester_5/Stage/StageOpdracht/Master


In [2]:
import os
import time

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

try:
    import pandas as pd
except Exception:
    pd = None

from app.io.dicom_loader import load_dicom_series
from app.pipeline.ai_pipeline import PipelineConfig
from app.pipeline.echo_ocr_pipeline import EchoOcrPipeline
from app.pipeline.layout.echo_ocr_box_detector import TopLeftBlueGrayBoxDetector
from app.pipeline.measurements.line_first_parser import LineFirstParser
from app.pipeline.measurements.whole_blob_line_recovery import recover_lines_from_whole_blob_ocr
from app.pipeline.ocr.ocr_engines import UnavailableOcrEngineError, build_engine
from app.tools.batch.sweep_preprocessing_headless import _broad_configs, _build_preprocess_views
from app.validation.datasets import canonicalize_label_line, parse_labels
from app.validation.evaluation import score_predictions

In [3]:
LABELS_JSON = REPO / "labels" / "labels.json"
CONFIG_NAME = "gray_x3_lanczos"
ENGINE_NAME = "glm-ocr"
SPLIT = "validation"
ONLY_NAMES: set[str] | None = None  # e.g. {"92290733_0035.dcm"}
MAX_FILES: int | None = None        # e.g. 25 for a quick pass
FAILURE_PREVIEW_LIMIT = 25
DIAG_NAME: str | None = None
EXTERNAL_DICOM_ROOT = Path("/run/media/warre/T7/MIMIC-IV-ECHO/files")

if not LABELS_JSON.is_file():
    raise FileNotFoundError(f"Missing labels file: {LABELS_JSON}")
if EXTERNAL_DICOM_ROOT.is_dir():
    os.environ["ECHO_OCR_DICOM_ROOT"] = str(EXTERNAL_DICOM_ROOT)
    print(f"Using ECHO_OCR_DICOM_ROOT={EXTERNAL_DICOM_ROOT}")

sweep_cfg = next(cfg for cfg in _broad_configs() if cfg.name == CONFIG_NAME)
preprocess_views = _build_preprocess_views(sweep_cfg)
preprocess_full_roi = preprocess_views["default"]

detector = TopLeftBlueGrayBoxDetector()
text_parser = LineFirstParser()

try:
    whole_blob_engine = build_engine(ENGINE_NAME)
    segmented_pipe = EchoOcrPipeline(
        ocr_engine=whole_blob_engine,
        config=PipelineConfig(
            parameters={
                "ocr_engine": ENGINE_NAME,
                "requested_ocr_engine": ENGINE_NAME,
                "parser_mode": "off",
            }
        ),
    )
    segmented_pipe.ensure_components()
    segmented_pipe._line_transcriber.preprocess_views = preprocess_views
except UnavailableOcrEngineError as exc:
    raise RuntimeError(f"Could not start OCR engine {ENGINE_NAME!r}: {exc}") from exc

labeled_files = parse_labels(LABELS_JSON, split_filter={SPLIT})
if ONLY_NAMES is not None:
    labeled_files = [item for item in labeled_files if item.file_name in ONLY_NAMES]
if MAX_FILES is not None:
    labeled_files = labeled_files[: int(MAX_FILES)]

labeled_by_name = {item.file_name: item for item in labeled_files}

print(f"Loaded {len(labeled_files)} labeled files from split={SPLIT!r}")
print(f"Preprocess config: {CONFIG_NAME}")
print(f"OCR engine: {ENGINE_NAME}")

Using ECHO_OCR_DICOM_ROOT=/run/media/warre/T7/MIMIC-IV-ECHO/files


The image processor of type `Glm46VImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loaded 353 labeled files from split='validation'
Preprocess config: gray_x3_lanczos
OCR engine: glm-ocr


In [4]:
STRATEGIES = ("blob_parser", "blob_recovered", "segmented_pipeline")


def measurement_to_prediction(measurement) -> dict[str, str | None]:
    return {
        "name": getattr(measurement, "name", None),
        "value": getattr(measurement, "value", None),
        "unit": getattr(measurement, "unit", None),
    }


def prediction_to_line(prediction: dict[str, str | None]) -> str:
    parts = [
        str(prediction.get("name") or "").strip(),
        str(prediction.get("value") or "").strip(),
        str(prediction.get("unit") or "").strip(),
    ]
    return canonicalize_label_line(" ".join(part for part in parts if part))


def display_table(rows: list[dict], *, limit: int | None = None) -> None:
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        display(pd.DataFrame(shown))
        return
    for row in shown:
        print(row)


def finalize_strategy_result(labeled_file, *, predictions: list[dict[str, str | None]], raw_ocr_texts: list[str], recovered_lines: list[str], recovery_source: str, detected_frame_count: int, elapsed_s: float) -> dict[str, object]:
    match_results = score_predictions(labeled_file.measurements, predictions)
    return {
        "file_name": labeled_file.file_name,
        "file_path": str(labeled_file.path),
        "split": labeled_file.split,
        "labels": [item.text for item in labeled_file.measurements],
        "predicted_lines": [prediction_to_line(item) for item in predictions],
        "raw_ocr_text": "\n\n".join(raw_ocr_texts),
        "recovered_lines": list(recovered_lines),
        "recovery_source": recovery_source,
        "total_labels": len(labeled_file.measurements),
        "full_matches": sum(1 for item in match_results if item.full_match),
        "line_matches": sum(1 for item in match_results if item.line_match),
        "value_matches": sum(1 for item in match_results if item.value_match),
        "label_matches": sum(1 for item in match_results if item.label_match),
        "prefix_matches": sum(1 for item in match_results if item.prefix_match),
        "predicted_count": len(predictions),
        "detected_frame_count": detected_frame_count,
        "elapsed_s": elapsed_s,
    }


def evaluate_file_all_strategies(labeled_file) -> dict[str, dict[str, object]]:
    started = time.perf_counter()
    series = load_dicom_series(labeled_file.path, load_pixels=True)

    blob_parser_predictions: list[dict[str, str | None]] = []
    blob_parser_texts: list[str] = []
    blob_parser_detected_frames = 0

    blob_recovered_predictions: list[dict[str, str | None]] = []
    blob_recovered_texts: list[str] = []
    blob_recovered_lines: list[str] = []
    blob_recovery_sources: list[str] = []
    blob_recovered_detected_frames = 0

    segmented_predictions: list[dict[str, str | None]] = []
    segmented_texts: list[str] = []
    segmented_lines: list[str] = []
    segmented_detected_frames = 0

    for frame_index in range(series.frame_count):
        frame = series.get_frame(frame_index)
        detection = detector.detect(frame)
        if detection.present and detection.bbox is not None:
            x, y, bw, bh = detection.bbox
            roi = frame[y : y + bh, x : x + bw]
            prepared = preprocess_full_roi(roi)
            ocr_result = whole_blob_engine.extract(prepared)

            blob_parser_measurements = text_parser.parse(
                ocr_result.text,
                confidence=max(float(ocr_result.confidence), 1e-6),
            )
            blob_parser_predictions.extend(measurement_to_prediction(item) for item in blob_parser_measurements)
            blob_parser_texts.append(f"[frame {frame_index}]\n{ocr_result.text}")
            blob_parser_detected_frames += 1

            recovery = recover_lines_from_whole_blob_ocr(
                roi,
                ocr_result,
                confidence=max(float(ocr_result.confidence), 1e-6),
            )
            blob_recovered_predictions.extend(measurement_to_prediction(item) for item in recovery.measurements)
            blob_recovered_texts.append(f"[frame {frame_index}]\n{ocr_result.text}")
            blob_recovered_lines.extend(line.text for line in recovery.recovered_lines)
            blob_recovery_sources.append(str(recovery.debug.get("source") or ""))
            blob_recovered_detected_frames += 1

        _det, segmentation, segmented_ocr, panel, segmented_measurements, bbox = segmented_pipe.analyze_frame_with_debug(frame)
        if bbox is not None:
            segmented_predictions.extend(measurement_to_prediction(item) for item in segmented_measurements)
            if segmented_ocr is not None and segmented_ocr.text.strip():
                segmented_texts.append(f"[frame {frame_index}]\n{segmented_ocr.text}")
            segmented_lines.extend(line.text for line in panel.lines)
            segmented_detected_frames += 1

    elapsed_s = time.perf_counter() - started
    return {
        "blob_parser": finalize_strategy_result(
            labeled_file,
            predictions=blob_parser_predictions,
            raw_ocr_texts=blob_parser_texts,
            recovered_lines=[],
            recovery_source="raw_parser",
            detected_frame_count=blob_parser_detected_frames,
            elapsed_s=elapsed_s,
        ),
        "blob_recovered": finalize_strategy_result(
            labeled_file,
            predictions=blob_recovered_predictions,
            raw_ocr_texts=blob_recovered_texts,
            recovered_lines=blob_recovered_lines,
            recovery_source=", ".join(sorted({src for src in blob_recovery_sources if src})),
            detected_frame_count=blob_recovered_detected_frames,
            elapsed_s=elapsed_s,
        ),
        "segmented_pipeline": finalize_strategy_result(
            labeled_file,
            predictions=segmented_predictions,
            raw_ocr_texts=segmented_texts,
            recovered_lines=segmented_lines,
            recovery_source="segmented_pipeline",
            detected_frame_count=segmented_detected_frames,
            elapsed_s=elapsed_s,
        ),
    }

In [5]:
results_by_strategy = {name: [] for name in STRATEGIES}
missing_file_errors: list[dict[str, str]] = []
runtime_errors: list[dict[str, str]] = []

for index, labeled_file in enumerate(labeled_files, start=1):
    if index == 1 or index % 25 == 0 or index == len(labeled_files):
        print(f"[{index}/{len(labeled_files)}] {labeled_file.file_name}")
    try:
        file_results = evaluate_file_all_strategies(labeled_file)
        for strategy_name in STRATEGIES:
            results_by_strategy[strategy_name].append(file_results[strategy_name])
    except Exception as exc:
        err = {
            "file_name": labeled_file.file_name,
            "file_path": str(labeled_file.path),
            "error": repr(exc),
        }
        if "File not found" in err["error"]:
            missing_file_errors.append(err)
        else:
            runtime_errors.append(err)

summary_rows: list[dict[str, object]] = []
for strategy_name in STRATEGIES:
    rows = results_by_strategy[strategy_name]
    total_labels = sum(int(item["total_labels"]) for item in rows)
    exact_matches = sum(int(item["full_matches"]) for item in rows)
    line_matches = sum(int(item["line_matches"]) for item in rows)
    value_matches = sum(int(item["value_matches"]) for item in rows)
    summary_rows.append(
        {
            "strategy": strategy_name,
            "files_processed": len(rows),
            "files_missing": len(missing_file_errors),
            "runtime_errors": len(runtime_errors),
            "files_with_detected_roi": sum(1 for item in rows if int(item["detected_frame_count"]) > 0),
            "total_labels": total_labels,
            "exact_matches": exact_matches,
            "exact_errors": total_labels - exact_matches,
            "exact_match_rate": round(exact_matches / max(total_labels, 1), 4),
            "line_match_rate": round(line_matches / max(total_labels, 1), 4),
            "value_match_rate": round(value_matches / max(total_labels, 1), 4),
            "predicted_measurements": sum(int(item["predicted_count"]) for item in rows),
            "mean_seconds_per_file": round(
                sum(float(item["elapsed_s"]) for item in rows) / max(len(rows), 1),
                3,
            ),
        }
    )

blob_parser_by_name = {item["file_name"]: item for item in results_by_strategy["blob_parser"]}
blob_recovered_by_name = {item["file_name"]: item for item in results_by_strategy["blob_recovered"]}
segmented_by_name = {item["file_name"]: item for item in results_by_strategy["segmented_pipeline"]}

comparison_rows: list[dict[str, object]] = []
for file_name, parser_result in blob_parser_by_name.items():
    recovered_result = blob_recovered_by_name[file_name]
    segmented_result = segmented_by_name[file_name]
    comparison_rows.append(
        {
            "file_name": file_name,
            "labels": parser_result["total_labels"],
            "blob_parser_exact": parser_result["full_matches"],
            "blob_recovered_exact": recovered_result["full_matches"],
            "segmented_exact": segmented_result["full_matches"],
            "recovered_minus_parser": int(recovered_result["full_matches"]) - int(parser_result["full_matches"]),
            "recovered_minus_segmented": int(recovered_result["full_matches"]) - int(segmented_result["full_matches"]),
            "recovery_source": recovered_result["recovery_source"],
        }
    )

comparison_rows.sort(
    key=lambda row: (
        -int(row["recovered_minus_parser"]),
        int(row["recovered_minus_segmented"]),
        str(row["file_name"]),
    )
)
positive_gain_rows = [row for row in comparison_rows if int(row["recovered_minus_parser"]) > 0]
regression_rows = [row for row in comparison_rows if int(row["recovered_minus_parser"]) < 0]

print("Strategy summary")
display_table(summary_rows)
print(f"Missing files: {len(missing_file_errors)}")
print(f"Runtime errors: {len(runtime_errors)}")

[1/353] 91243943_0004.dcm
[25/353] 91629559_0027.dcm
[50/353] 92290733_0071.dcm
[75/353] 93330659_0103.dcm
[100/353] 93489296_0068.dcm
[125/353] 94106955_0046.dcm
[150/353] 94134445_0042.dcm
[175/353] 95253372_0042.dcm
[200/353] 97273331_0029.dcm
[225/353] 97555635_0102.dcm
[250/353] 98667422_0037.dcm
[275/353] 99094104_0042.dcm
[300/353] 99583722_0015.dcm
[325/353] 99583722_0092.dcm
[350/353] 99663585_0107.dcm
[353/353] 99663585_0110.dcm
Strategy summary
{'strategy': 'blob_parser', 'files_processed': 353, 'files_missing': 0, 'runtime_errors': 0, 'files_with_detected_roi': 353, 'total_labels': 901, 'exact_matches': 696, 'exact_errors': 205, 'exact_match_rate': 0.7725, 'line_match_rate': 0.7725, 'value_match_rate': 0.8724, 'predicted_measurements': 809, 'mean_seconds_per_file': 2.286}
{'strategy': 'blob_recovered', 'files_processed': 353, 'files_missing': 0, 'runtime_errors': 0, 'files_with_detected_roi': 353, 'total_labels': 901, 'exact_matches': 845, 'exact_errors': 56, 'exact_match_r

In [6]:
print("Hybrid gains over raw whole-blob parser")
if positive_gain_rows:
    display_table(positive_gain_rows, limit=FAILURE_PREVIEW_LIMIT)
else:
    print("No positive gains in this run.")

print("Worst hybrid regressions vs raw whole-blob parser")
display_table(regression_rows, limit=FAILURE_PREVIEW_LIMIT)

hybrid_failures = [
    {
        "file_name": item["file_name"],
        "total_labels": item["total_labels"],
        "exact_matches": item["full_matches"],
        "exact_errors": int(item["total_labels"]) - int(item["full_matches"]),
        "predicted_lines": " | ".join(item["predicted_lines"]),
        "recovered_lines": " | ".join(item["recovered_lines"]),
        "raw_ocr_text": item["raw_ocr_text"],
        "recovery_source": item["recovery_source"],
    }
    for item in results_by_strategy["blob_recovered"]
    if int(item["full_matches"]) < int(item["total_labels"])
]
hybrid_failures.sort(key=lambda row: (-int(row["exact_errors"]), str(row["file_name"])))

print("Hybrid failure preview")
display_table(hybrid_failures, limit=FAILURE_PREVIEW_LIMIT)

if missing_file_errors:
    print("Missing file paths")
    display_table(missing_file_errors, limit=FAILURE_PREVIEW_LIMIT)
if runtime_errors:
    print("Runtime errors")
    display_table(runtime_errors, limit=FAILURE_PREVIEW_LIMIT)

Hybrid gains over raw whole-blob parser
{'file_name': '91243943_0033.dcm', 'labels': 3, 'blob_parser_exact': 0, 'blob_recovered_exact': 3, 'segmented_exact': 3, 'recovered_minus_parser': 3, 'recovered_minus_segmented': 0, 'recovery_source': 'unit_boundary_split'}
{'file_name': '91629559_0041.dcm', 'labels': 3, 'blob_parser_exact': 0, 'blob_recovered_exact': 3, 'segmented_exact': 3, 'recovered_minus_parser': 3, 'recovered_minus_segmented': 0, 'recovery_source': 'unit_boundary_split'}
{'file_name': '93489296_0069.dcm', 'labels': 3, 'blob_parser_exact': 0, 'blob_recovered_exact': 3, 'segmented_exact': 3, 'recovered_minus_parser': 3, 'recovered_minus_segmented': 0, 'recovery_source': 'unit_boundary_split'}
{'file_name': '94106955_0051.dcm', 'labels': 3, 'blob_parser_exact': 0, 'blob_recovered_exact': 3, 'segmented_exact': 3, 'recovered_minus_parser': 3, 'recovered_minus_segmented': 0, 'recovery_source': 'unit_boundary_split'}
{'file_name': '94134445_0044.dcm', 'labels': 3, 'blob_parser_exa

In [7]:
def show_diagnostics(file_name: str) -> None:
    parser_result = blob_parser_by_name[file_name]
    recovered_result = blob_recovered_by_name[file_name]
    segmented_result = segmented_by_name[file_name]

    print(file_name)
    print("Expected labels:")
    for line in parser_result["labels"]:
        print("  -", line)

    print("\nWhole blob raw parser:")
    for line in parser_result["predicted_lines"]:
        print("  -", line)
    print(parser_result["raw_ocr_text"] or "<empty>")

    print("\nWhole blob recovered lines:")
    for line in recovered_result["recovered_lines"]:
        print("  -", line)
    print(f"recovery_source={recovered_result['recovery_source']}")

    print("\nSegmented pipeline lines:")
    for line in segmented_result["recovered_lines"]:
        print("  -", line)


if DIAG_NAME is None and comparison_rows:
    DIAG_NAME = str(comparison_rows[0]["file_name"])

if DIAG_NAME is not None:
    show_diagnostics(DIAG_NAME)
else:
    print("No files to inspect.")

91243943_0033.dcm
Expected labels:
  - 1 LALs A4C 6.1 cm
  - LAAs A4C 23.4 cm2
  - LAESV A-L A4C 77 ml

Whole blob raw parser:
  - 1 LALs A4C 6.1 cm LAAs A4C 23.4 cm2 LAESV A-L A4C 77 ml
[frame 0]
1 LALs A4C 6.1 cm LAAs A4C 23.4 cm2 LAESV A-L A4C 77 ml

Whole blob recovered lines:
  - 1 LALs A4C 6.1 cm
  - LAAs A4C 23.4 cm2
  - LAESV A-L A4C 77 ml
recovery_source=unit_boundary_split

Segmented pipeline lines:
  - 
  - 1 LALs A4C 6.1 cm
  - LAAs A4C 23.4 cm2
  - LAESV A-L A4C 77 ml
